# Multi-Region Cell Segmentation Pipeline
# Sequential Processing: Scripts 6a → 6b → 6c

## Overview
This pipeline performs advanced multi-region segmentation of FRET cells by identifying different cellular compartments based on brightness patterns and vesicle detection. The analysis separates cells into multiple functional regions to study heterogeneous protein distributions and dynamics.

## Biological Background
Membrone proteins often show:
- **Brightness heterogeneity**: Different molecular aggregation states
- **Mobile vesicles**: Fast-moving organelles in red-delay channel  
- **Compartmentalization**: Distinct regions with different protein concentrations

## Pipeline Structure
1. **Script 6a**: Export red-delay time series for vesicle detection
2. **Script 6b**: Combine probability maps from external analysis (Ilastik)
3. **Script 6c**: Generate multiple segmentation masks using two approaches

## Input/Output Summary
- **Input**: `*.ptu` files, N&B analysis results, Ilastik probability maps
- **Output**: Multiple region masks (`*_vesicles.tif`, `*_low.tif`, `*_high.tif`, `*_lowNB.tif`, `*_highNB.tif`)

## External Dependencies
- **Ilastik software**: For vesicle detection using "cell counting" approach
- **tttrlib**: For TTTR data processing

---


## SCRIPT 6A: EXPORT RED-DELAY TIME SERIES


## Red-Delay Time Series Export Pipeline

### Overview
Extracts red-delay channel time series from TTTR (.ptu) files for subsequent vesicle detection analysis in Ilastik. The red-delay channel is particularly sensitive to mobile vesicles and organelles.

### Input/Output
- **Input**: `*.ptu` files (time-tagged time-resolved data)
- **Output**: Individual frame TIFF files (`*_red_delay_XXX.tif`) in `./Cells/Red_delay/`

### Configuration Parameters
- **Channels**: Red s-polarized (ch1) and p-polarized (ch2)  
- **Time window**: 2500-5000 ch (delay range for acceptor detection, 25-50 ns)
- **Output format**: 16-bit TIFF files

### Processing Steps
1. Load TTTR data from .ptu files
2. Extract red delay channel intensities (s + p polarization)
3. Export each time frame as separate TIFF file
4. Prepare data for Ilastik vesicle detection

In [1]:
# Import required libraries
import glob
import os
import tifffile as tiff
import tttrlib

In [2]:
# Configuration parameters
path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA*/*.ptu'        # Input file pattern, only FRET samples to be processed
output_dir = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/'    # Output directory for individual frames

In [3]:
# Channel assignments for red detection
red_s_ch = 1    # Red s-polarized channel
red_p_ch = 2    # Red p-polarized channel

# Time window for delay detection (in channels)
# This captures the delayed fluorescence component
delay_range = 2500, 5000

In [4]:
# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Process each TTTR file in batch
for file in glob.glob(path):
    # Extract filename without extension
    filename = os.path.splitext(os.path.basename(file))[0]
    
    # Load TTTR data
    data = tttrlib.TTTR(file)
    clsm_image = tttrlib.CLSMImage(tttr_data=data)

    print(f'Processing.... {filename}')

    # Extract red s-polarized delay intensities  
    clsm_image.fill(data, channels=[red_s_ch], micro_time_ranges=[delay_range])
    int_red_s_delay = clsm_image.intensity
    
    # Extract red p-polarized delay intensities
    clsm_image.fill(data, channels=[red_p_ch], micro_time_ranges=[delay_range])
    int_red_p_delay = clsm_image.intensity

    # Combine both polarizations for total red-delay signal
    SUM_red_delay = int_red_s_delay + int_red_p_delay

    # Export each time frame as individual TIFF file
    # This format is required for Ilastik batch processing
    for frame_idx in range(SUM_red_delay.shape[0]):  # Loop through time frames
        save_path = os.path.join(output_dir, f"{filename}_red_delay_{frame_idx:03d}.tif")
        tiff.imwrite(save_path, SUM_red_delay[frame_idx].astype('uint16'))  # Save as 16-bit TIFF

    print(f"Saved {SUM_red_delay.shape[0]} images in {output_dir} for {filename}")

Processing.... low_FRET_1-5_64
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_64
Processing.... low_FRET_1-5_66
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_66
Processing.... low_FRET_1-5_68
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_68
Processing.... low_FRET_1-5_70
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_70
Processing.... low_FRET_1-5_72
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_72
Processing.... low_FRET_1-5_74
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_74
Processing.... low_FRET_1-5_76
Saved 50 images in C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Red delay/ for low_FRET_1-5_76
